# Polarity (signed): excitation − inhibition

For each path length `k=1..4`: `find_paths_of_length` → `group_paths` (instance-level) → `signed_effective_conn_from_paths` → `tmp_e - tmp_i`. Sum across `k`, row-normalize by absolute-value sum (so Σ|bars| = 1 per target), predict `ON if L1_R > L2_R else OFF`. Sign overrides: `L1_R → +1` (flip), `sign == 0 → +1`. Plots all three panels (VPN, non-VPN OL, CB) as signed stacked bars.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'make_figures' else Path.cwd().resolve()))

import numpy as np
import pandas as pd
import scipy.sparse as sp
import plotly.express as px

from utils import config
from utils.palettes import load_colors
from utils import core_data, polarity
from connectome_interpreter.path_finding import find_paths_of_length, group_paths
from connectome_interpreter import signed_effective_conn_from_paths

## Helpers

In [ ]:
NR_HOPS_MAX = 4


def build_sign_map(meta_df):
    """Instance → ±1 with overrides: L1_R flipped to +1, sign==0 coerced to +1."""
    s = meta_df[['instance', 'sign']].drop_duplicates('instance').set_index('instance')['sign'].copy()
    if 'L1_R' in s.index:
        s.loc['L1_R'] = 1
    s[s == 0] = 1
    return s.astype(int).to_dict()


def compute_signed_totals(A, inidx, outidx, idx_to_instance, inst_to_sign,
                           nr_hops_max=NR_HOPS_MAX):
    total = None
    for k in range(1, nr_hops_max + 1):
        raw = find_paths_of_length(A, inidx, outidx, k, quiet=True)
        if raw is None or raw.empty:
            continue
        paths = group_paths(raw, idx_to_instance, idx_to_instance, combining_method='mean')
        tmp_e, tmp_i = signed_effective_conn_from_paths(paths, wide=True, idx_to_nt=inst_to_sign)
        tmp = tmp_e.sub(tmp_i, fill_value=0)
        print(f'  k={k}: {len(raw):,} path-edges, signed table {tmp.shape}')
        total = tmp if total is None else total.add(tmp, fill_value=0)
    return total


def build_cmp(total_signed, labels_df, seed_instance_list, meta_df):
    """Row-normalized (absolute-value) signed feature table joined with polarity labels.
    Adds `main_groups`, `predicted`, `match` columns."""
    feat = total_signed.T
    feat = feat.reindex(index=labels_df['instance'].values,
                        columns=list(seed_instance_list), fill_value=0).fillna(0)
    abs_sum = feat.abs().sum(axis=1)
    feat_norm = feat.div(abs_sum.where(abs_sum > 0, 1), axis=0).reset_index(names='instance')
    merged = labels_df.merge(feat_norm, on='instance', how='inner').merge(
        meta_df[['instance', 'main_groups']].drop_duplicates('instance'),
        on='instance', how='left',
    )
    l1 = merged['L1_R'] if 'L1_R' in merged.columns else 0.0
    l2 = merged['L2_R'] if 'L2_R' in merged.columns else 0.0
    merged['predicted'] = np.where(l1 > l2, 'ON', 'OFF')
    merged['match'] = merged['predicted'] == merged['polarity']
    return merged


def plot_signed_bars(cmp_df, colored_seed, filename_stem, width_scale=1.0):
    from utils.plotting import _POLARITY_DISPLAY_RENAMES

    suffix = '_R'
    seed_cols = [s + suffix for s in colored_seed.index if s + suffix in cmp_df.columns]
    long = cmp_df[['instance'] + seed_cols].copy()
    long['out type'] = long['instance'].str[:-2]
    melted = long.melt(id_vars=['out type'], value_vars=seed_cols,
                        var_name='in type', value_name='effective weight')
    melted['in type'] = melted['in type'].str[:-2]
    cmap = colored_seed['color'].to_dict()
    width = int(max(400, 35 * len(cmp_df) + 150) * 0.8 * width_scale)
    fig = px.bar(melted, x='out type', y='effective weight', color='in type',
                 color_discrete_map=cmap, category_orders={'in type': list(cmap.keys())})
    fig.add_hline(y=0, line_color='black', line_width=1)
    fig.add_hline(y=1, line_color='black', line_width=1)
    fig.add_hline(y=-1, line_color='black', line_width=1)
    fig.update_layout(xaxis_tickangle=-45, yaxis_range=[-1.05, 1.05],
                       font=dict(family='arial', size=18), height=350, width=width,
                       plot_bgcolor='white', paper_bgcolor='white')
    order = cmp_df['instance'].str[:-2].tolist()
    labels_x = [
        f'<b>{_POLARITY_DISPLAY_RENAMES.get(t, t)}</b>' if p == 'OFF'
        else _POLARITY_DISPLAY_RENAMES.get(t, t)
        for t, p in zip(order, cmp_df['polarity'])
    ]
    fig.update_xaxes(showgrid=False, categoryorder='array', categoryarray=order,
                      tickvals=order, ticktext=labels_x)
    fig.update_yaxes(showgrid=False, tickvals=[-1, 0, 1])
    out_dir = config.FIG_DIR / 'exp_comparison'
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = out_dir / f'{config.FIG_DATASET}_{filename_stem}'
    fig.write_image(str(stem) + '.pdf')
    fig.write_html(str(stem) + '.html')
    print('wrote', str(stem) + '.pdf')
    return fig


## OL subset — VPN (OL output) + non-VPN (`_other`)

In [ ]:
ol_meta = core_data.get_ol_meta()
flow_type = core_data.get_ol_flow_type()
labels_ol = polarity.get_polarity_experiments()
labels_ol = labels_ol[labels_ol['in_ol']].reset_index(drop=True)

A_OL = sp.load_npz(config.DATA_DIR / f'{config.DATASET}_OL_r_lat_flow_0.npz').tocsr()

in_instances_ol = flow_type[flow_type.hitting_time == 0].instance.unique()
inidx_ol  = ol_meta[ol_meta.instance.isin(in_instances_ol)].idx.values
outidx_ol = ol_meta[ol_meta.instance.isin(labels_ol['instance'].values)].idx.values
idx_to_instance_ol = dict(zip(ol_meta.idx, ol_meta.instance))
inst_to_sign_ol = build_sign_map(ol_meta)

total_ol = compute_signed_totals(
    A_OL, inidx_ol, outidx_ol, idx_to_instance_ol, inst_to_sign_ol,
)
print('total_ol shape:', total_ol.shape)

In [ ]:
_, _, _, colored_seed = load_colors()

cmp_ol = build_cmp(total_ol, labels_ol, in_instances_ol, ol_meta)
cmp_ol = cmp_ol[cmp_ol['main_groups'] != 'visual input'].reset_index(drop=True)

cmp_vpn   = cmp_ol[cmp_ol['main_groups'] == 'OL output'].reset_index(drop=True)
cmp_other = cmp_ol[cmp_ol['main_groups'] != 'OL output'].reset_index(drop=True)

for label, df in [('VPN', cmp_vpn), ('non-VPN OL', cmp_other)]:
    miss = df[~df.match][['instance', 'polarity', 'predicted', 'main_groups']]
    print(f'{label}: {df.match.mean():.1%} match ({len(df)} types, {len(miss)} mismatches)')
    if len(miss):
        print(miss.to_string(index=False))

plot_signed_bars(cmp_vpn,   colored_seed, 'polarity_predictions_vpn_signed')
plot_signed_bars(cmp_other, colored_seed, 'polarity_predictions_other_signed', width_scale=0.88)


## CB (full brain)

In [ ]:
meta = core_data.get_meta()
labels_cb = polarity.get_polarity_experiments()
labels_cb = labels_cb[~labels_cb['in_ol']].reset_index(drop=True)

A_full = sp.load_npz(config.DATA_DIR / f'{config.DATASET}_r_lat_flow_0.npz').tocsr()
print(f'full-brain A: {A_full.shape}, nnz={A_full.nnz}')

in_instances_cb = meta[(meta.main_groups == 'visual input') & (meta.side == 'right')].instance.unique()
inidx_cb  = meta[meta.instance.isin(in_instances_cb)].idx.values
outidx_cb = meta[meta.instance.isin(labels_cb['instance'].values)].idx.values
idx_to_instance_full = dict(zip(meta.idx, meta.instance))
inst_to_sign_full = build_sign_map(meta)

total_cb = compute_signed_totals(
    A_full, inidx_cb, outidx_cb, idx_to_instance_full, inst_to_sign_full,
)
print('total_cb shape:', total_cb.shape)

In [ ]:
cmp_cb = build_cmp(total_cb, labels_cb, in_instances_cb, meta)

miss_cb = cmp_cb[~cmp_cb.match][['instance', 'polarity', 'predicted', 'main_groups']]
print(f'CB: {cmp_cb.match.mean():.1%} match ({len(cmp_cb)} types, {len(miss_cb)} mismatches)')
if len(miss_cb):
    print(miss_cb.to_string(index=False))

plot_signed_bars(cmp_cb, colored_seed, 'polarity_predictions_cb_signed')